# Notebook 06: Baseline RAG Answer Generation

## Purpose
This notebook implements the first end-to-end grounded answer generation stage for Turkish legal RAG.

**Pipeline:**
```
Query → Dense Embedding + BM25 → Hybrid Fusion → Cross-Encoder Reranking → LLM Answer Generation
```

**Scope:**
- Load retrieval artifacts from previous stages (notebooks 01-05)
- Reuse existing hybrid + reranker pipeline to retrieve top context chunks
- Load a generation model (Gemma-2-9B-IT by default)
- Build retrieval-aware prompts that force grounded answers
- Generate answers for test Turkish legal queries
- Save results with source attribution

**Output:** `rag_generated_answers.csv`, `rag_generated_answers.jsonl`, metadata

---

**Next Steps After This Notebook:**
- Notebook 07: Benchmark evaluation (retrieval metrics, answer quality, hallucination)
- Notebook 08: Gradio interface for interactive demo
- Fine-tune reranker for Turkish-legal domain
- Fine-tune or prompt-engineer generation model for better legal answers

# Section 1: Environment Setup

In [ ]:
# Google Colab environment check
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python version: {sys.version}")

# Section 2: Google Drive Mount

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive")
else:
    print("Not in Colab - skipping Drive mount")

# Section 3: Configurable Paths and Parameters

**UPDATE THESE IF YOUR PATHS DIFFER:**

In [ ]:
import os

# ===================== PATHS =====================
PROJECT_ROOT = "/content/drive/My Drive/nlp-rag-project"

RETRIEVAL_DATA_PATH = f"{PROJECT_ROOT}/data/retrieval/retrieval_corpus_full.csv"
DENSE_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/dense_retrieval"
HYBRID_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/hybrid_retrieval"
RERANKER_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/reranker"
RAG_OUTPUT_DIR = f"{PROJECT_ROOT}/outputs/rag_generation"

# Create RAG output directory if not exists
os.makedirs(RAG_OUTPUT_DIR, exist_ok=True)

# ===================== MODEL NAMES =====================
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATION_MODEL_NAME = "google/gemma-2-9b-it"  # Can change to "google/gemma-2-2b-it" for smaller GPU

# ===================== RETRIEVAL PARAMETERS =====================
TOP_K_FINAL = 3  # Number of sources to use for answer generation
DENSE_CANDIDATES = 20
BM25_CANDIDATES = 20
HYBRID_CANDIDATES = 20
ALPHA = 0.5  # Weight blend: 0.5 = equal dense + BM25

# ===================== GENERATION PARAMETERS =====================
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2  # Low temp = more focused, deterministic
TOP_P = 0.9
TOP_K = 40

print(f"Project root: {PROJECT_ROOT}")
print(f"RAG output directory: {RAG_OUTPUT_DIR}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Reranker model: {RERANKER_MODEL_NAME}")
print(f"Generation model: {GENERATION_MODEL_NAME}")
print(f"\nRetrieval config: Top-K={TOP_K_FINAL}, Alpha={ALPHA}")
print(f"Generation config: MaxTokens={MAX_NEW_TOKENS}, Temp={TEMPERATURE}")

# Section 4: Install and Import Dependencies

In [ ]:
# Install required packages
!pip install -q pandas numpy
!pip install -q faiss-cpu
!pip install -q rank-bm25
!pip install -q sentence-transformers
!pip install -q transformers accelerate torch

print("Dependencies installed.")

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("All dependencies imported successfully.")

# Section 5: Load Retrieval Corpus

In [ ]:
# Load the full retrieval corpus
print(f"Loading retrieval corpus from: {RETRIEVAL_DATA_PATH}")
retrieval_corpus = pd.read_csv(RETRIEVAL_DATA_PATH)

print(f"Corpus shape: {retrieval_corpus.shape}")
print(f"\nColumns: {retrieval_corpus.columns.tolist()}")
print(f"\nFirst few rows:")
print(retrieval_corpus.head(3))

# Verify required columns
required_cols = ['chunk_id', 'chunk_text', 'source', 'question', 'answer']
for col in required_cols:
    if col not in retrieval_corpus.columns:
        print(f"WARNING: Missing required column: {col}")
    else:
        print(f"✓ Column '{col}' present")

# Section 6: Load Dense Retrieval Artifacts

In [ ]:
# Load dense retrieval artifacts
print("Loading dense retrieval artifacts...\n")

# Load embeddings
embeddings_path = f"{DENSE_OUTPUT_DIR}/corpus_embeddings.npy"
corpus_embeddings = np.load(embeddings_path)
print(f"Embeddings shape: {corpus_embeddings.shape}")

# Load FAISS index
faiss_index_path = f"{DENSE_OUTPUT_DIR}/faiss_index.bin"
faiss_index = faiss.read_index(faiss_index_path)
print(f"FAISS index loaded, ntotal={faiss_index.ntotal}")

# Load row mapping (maps FAISS index to corpus row)
row_mapping_path = f"{DENSE_OUTPUT_DIR}/retrieval_row_mapping.csv"
row_mapping = pd.read_csv(row_mapping_path)
print(f"Row mapping shape: {row_mapping.shape}")
print(f"\nRow mapping sample:")
print(row_mapping.head())

# Section 7: Load Retrieval Models

In [ ]:
# Load embedding model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Embedding model loaded. Output dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Load reranker model
print(f"\nLoading reranker model: {RERANKER_MODEL_NAME}")
reranker = CrossEncoder(RERANKER_MODEL_NAME)
print(f"Reranker model loaded.")

# Section 8: Retrieval Helper Functions

In [ ]:
def retrieve_dense_candidates(
    query: str,
    embedding_model: SentenceTransformer,
    faiss_index,
    retrieval_corpus: pd.DataFrame,
    row_mapping: pd.DataFrame,
    k: int
) -> pd.DataFrame:
    """Retrieve dense candidates using FAISS."""
    # Encode query
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    query_embedding = query_embedding.reshape(1, -1).astype(np.float32)
    
    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)
    
    # Search
    distances, indices = faiss_index.search(query_embedding, k)
    scores = distances[0]
    indices = indices[0]
    
    # Map to corpus rows
    corpus_indices = row_mapping.loc[indices, 'corpus_row_index'].values
    results = retrieval_corpus.iloc[corpus_indices].copy()
    results['dense_score'] = scores
    
    return results.reset_index(drop=True)


def retrieve_bm25_candidates(
    query: str,
    retrieval_corpus: pd.DataFrame,
    k: int
) -> pd.DataFrame:
    """Retrieve BM25 candidates."""
    # Preprocess
    corpus_texts = retrieval_corpus['chunk_text'].fillna('').str.lower().str.split().tolist()
    bm25 = BM25Okapi(corpus_texts)
    
    # Tokenize query
    query_tokens = query.lower().split()
    
    # Score
    scores = bm25.get_scores(query_tokens)
    
    # Get top-k
    top_indices = np.argsort(scores)[::-1][:k]
    results = retrieval_corpus.iloc[top_indices].copy()
    results['bm25_score'] = scores[top_indices]
    
    return results.reset_index(drop=True)


def retrieve_hybrid_candidates(
    query: str,
    embedding_model: SentenceTransformer,
    faiss_index,
    retrieval_corpus: pd.DataFrame,
    row_mapping: pd.DataFrame,
    dense_k: int,
    bm25_k: int,
    alpha: float
) -> pd.DataFrame:
    """Retrieve hybrid candidates: dense + BM25 with fusion."""
    # Dense
    dense_results = retrieve_dense_candidates(
        query, embedding_model, faiss_index, retrieval_corpus, row_mapping, dense_k
    )
    
    # BM25
    bm25_results = retrieve_bm25_candidates(query, retrieval_corpus, bm25_k)
    
    # Merge and normalize
    merged = pd.concat([dense_results, bm25_results], ignore_index=True)
    merged = merged.drop_duplicates(subset=['chunk_id'], keep='first').reset_index(drop=True)
    
    # Fill missing scores
    merged['dense_score'] = merged['dense_score'].fillna(0.0)
    merged['bm25_score'] = merged['bm25_score'].fillna(0.0)
    
    # Normalize separately
    if merged['dense_score'].max() > 0:
        dense_norm = (merged['dense_score'] - merged['dense_score'].min()) / (merged['dense_score'].max() - merged['dense_score'].min())
    else:
        dense_norm = pd.Series(0.0, index=merged.index)
    
    if merged['bm25_score'].max() > 0:
        bm25_norm = (merged['bm25_score'] - merged['bm25_score'].min()) / (merged['bm25_score'].max() - merged['bm25_score'].min())
    else:
        bm25_norm = pd.Series(0.0, index=merged.index)
    
    # Fusion
    merged['hybrid_score'] = alpha * dense_norm + (1 - alpha) * bm25_norm
    merged = merged.sort_values('hybrid_score', ascending=False).reset_index(drop=True)
    
    return merged


def rerank_candidates(
    query: str,
    candidate_df: pd.DataFrame,
    reranker: CrossEncoder,
    top_k: int
) -> pd.DataFrame:
    """Rerank candidates using cross-encoder."""
    if len(candidate_df) == 0:
        return pd.DataFrame()
    
    chunks = candidate_df['chunk_text'].tolist()
    pairs = [[query, chunk] for chunk in chunks]
    scores = reranker.predict(pairs)
    
    result_df = candidate_df.copy()
    result_df['reranker_score'] = scores
    result_df = result_df.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    result_df['rank'] = range(1, len(result_df) + 1)
    
    return result_df.head(top_k)


def get_final_context(
    query: str,
    embedding_model: SentenceTransformer,
    faiss_index,
    retrieval_corpus: pd.DataFrame,
    row_mapping: pd.DataFrame,
    reranker: CrossEncoder,
    top_k: int,
    alpha: float
) -> pd.DataFrame:
    """
    Full retrieval pipeline: hybrid + reranker.
    Returns top-k reranked context for answer generation.
    """
    # Hybrid retrieval
    hybrid = retrieve_hybrid_candidates(
        query,
        embedding_model,
        faiss_index,
        retrieval_corpus,
        row_mapping,
        dense_k=DENSE_CANDIDATES,
        bm25_k=BM25_CANDIDATES,
        alpha=alpha
    )
    
    # Rerank
    reranked = rerank_candidates(query, hybrid.head(HYBRID_CANDIDATES), reranker, top_k)
    
    return reranked


print("Retrieval functions defined.")

# Section 9: Load Generation Model

In [ ]:
# Load tokenizer and model
print(f"Loading generation model: {GENERATION_MODEL_NAME}")
print(f"(This may take 1-2 minutes on first load)\n")

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

model = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded.")
print(f"Model dtype: {model.dtype}")
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print(f"No GPU available - will use CPU (slower)")

# Section 10: Build Retrieval-Aware Prompt

In [ ]:
def build_rag_prompt(
    query: str,
    context_chunks: pd.DataFrame,
    max_context_length: int = 2000
) -> Tuple[str, List[str]]:
    """
    Build a retrieval-aware prompt that forces grounded answers.
    
    Returns:
        (prompt, source_names): The full prompt and list of source names used
    """
    # System instruction
    system_instruction = """Sen Türk hukuk asistanısın. Sana verilen bağlam bilgisindeseniz kalmak zorundasın.
Dış bilgi ORANDEKİ PREMÜZLERİ KULLANAMAZSUR. Bağlam yetersiz veya belirsizse açıkça belirt.
Cevabını kısa, bilgilendirici ve hukuki tarz da verin. Sonunda destek kaynakları listele."""
    
    # Format context
    context_text = ""
    source_names = []
    used_count = 0
    
    for idx, row in context_chunks.iterrows():
        source = str(row.get('source', 'Bilinmeyen Kaynak')).strip()
        chunk = str(row.get('chunk_text', '')).strip()
        
        # Skip if source is NaN/null/empty
        if source.lower() in ['nan', 'none', 'bilinmeyen', '']:
            source = f"Kaynak {used_count + 1}"
        
        source_names.append(source)
        
        context_chunk = f"[Bağlam {used_count + 1}]\nKaynak: {source}\nMetin: {chunk}\n\n"
        
        if len(context_text + context_chunk) > max_context_length:
            break
        
        context_text += context_chunk
        used_count += 1
    
    # User prompt
    user_prompt = f"""Soru: {query}

{context_text}

Yukarıdaki bağlam bilgisini kullanarak soruyu cevapla. Yanıtın sonuna destek kaynakları ekle.
Bağlam yetersizse: "Verilen bağlamda yeterli bilgi yoktur." de."""
    
    # Combine system + user
    full_prompt = f"""<s>[INST] <<SYS>>
{system_instruction}
<</SYS>>

{user_prompt} [/INST]"""
    
    return full_prompt, source_names


print("RAG prompt builder defined.")

# Section 11: Answer Generation Function

In [ ]:
def generate_grounded_answer(
    query: str,
    context_chunks: pd.DataFrame,
    model,
    tokenizer,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE
) -> Dict:
    """
    Generate a grounded answer using retrieved context.
    
    Returns:
        {
            'query': str,
            'answer': str,
            'supporting_sources': List[str],
            'context_count': int,
            'model': str
        }
    """
    # Build prompt
    prompt, sources = build_rag_prompt(query, context_chunks)
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    
    # Move to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract answer (remove prompt from output)
    answer = generated_text[len(prompt):].strip()
    
    return {
        'query': query,
        'answer': answer,
        'supporting_sources': sources,
        'context_count': len(context_chunks),
        'model': GENERATION_MODEL_NAME
    }


print("Answer generation function defined.")

# Section 12: Test Queries

In [ ]:
# Same test queries as in previous notebooks
test_queries = [
    "Haksız zenginleşme ile ilgili hükümler nelerdir?",
    "Miras bırakanın tasarruf özgürlüğü nasıl sınırlandırılır?",
    "Ceza muhakemesinde tutuklama şartları nelerdir?",
    "Anayasa'ya göre devletin şekli nedir?",
    "Aile yurdu ile ilgili malik üzerindeki sınırlamalar nelerdir?"
]

print(f"Test queries ({len(test_queries)} total):\n")
for i, q in enumerate(test_queries, 1):
    print(f"{i}. {q}")

# Section 13: Run RAG on Test Queries

In [ ]:
print("Running RAG pipeline on test queries...\n")
print("=" * 80)

all_results = []

for test_idx, query in enumerate(test_queries, 1):
    print(f"\n[Query {test_idx}/{len(test_queries)}] {query}\n")
    
    # Retrieve context
    print("   → Retrieving context (hybrid + reranker)...")
    context = get_final_context(
        query,
        embedding_model,
        faiss_index,
        retrieval_corpus,
        row_mapping,
        reranker,
        top_k=TOP_K_FINAL,
        alpha=ALPHA
    )
    
    print(f"   → Retrieved {len(context)} context chunks")
    
    # Display top context sources
    print(f"\n   Top sources:")
    for rank_idx, row in context.iterrows():
        source = str(row.get('source', 'Bilinmeyen')).strip()
        if source.lower() in ['nan', 'none', '']:
            source = f"Kaynak {rank_idx + 1}"
        score = row.get('reranker_score', 0.0)
        chunk_preview = str(row.get('chunk_text', ''))[:80]+"..."
        print(f"      [{rank_idx+1}] ({score:.3f}) {source}: {chunk_preview}")
    
    # Generate answer
    print(f"\n   → Generating answer...")
    result = generate_grounded_answer(
        query,
        context,
        model,
        tokenizer
    )
    
    result['test_query_index'] = test_idx
    result['top_chunk_ids'] = context['chunk_id'].tolist()
    result['retrieval_model'] = EMBEDDING_MODEL_NAME
    result['reranker_model'] = RERANKER_MODEL_NAME
    
    all_results.append(result)
    
    # Display answer
    print(f"\n   ANSWER:")
    print(f"   {result['answer'][:500]}")
    if len(result['answer']) > 500:
        print(f"   [... {len(result['answer'])-500} more chars ...]")
    
    print(f"\n   Supporting sources: {result['supporting_sources'][:len(context)]}")
    print("\n" + "=" * 80)

print(f"\nCompleted RAG generation for all {len(test_queries)} queries.")

# Section 14: Save Generated Results

In [ ]:
# Convert results list to DataFrame
results_df = pd.DataFrame(all_results)

print(f"Results shape: {results_df.shape}")
print(f"Columns: {results_df.columns.tolist()}")
print(f"\nResults preview:")
print(results_df[['test_query_index', 'query', 'answer', 'context_count']].head())

In [ ]:
import json

jsonl_path = f"{RAG_OUTPUT_DIR}/rag_generated_answers.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for _, row in results_df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"✓ Saved JSONL: {jsonl_path}")

# Section 15: Save Metadata

In [ ]:
# Build metadata
metadata = {
    'timestamp': datetime.now().isoformat(),
    'num_test_queries': len(test_queries),
    'num_generated_answers': len(all_results),
    'retrieval_config': {
        'top_k_final': TOP_K_FINAL,
        'dense_candidates': DENSE_CANDIDATES,
        'bm25_candidates': BM25_CANDIDATES,
        'hybrid_candidates': HYBRID_CANDIDATES,
        'alpha_blend': ALPHA
    },
    'models': {
        'embedding_model': EMBEDDING_MODEL_NAME,
        'reranker_model': RERANKER_MODEL_NAME,
        'generation_model': GENERATION_MODEL_NAME
    },
    'generation_config': {
        'max_new_tokens': MAX_NEW_TOKENS,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'top_k': TOP_K
    },
    'corpus_size': len(retrieval_corpus),
    'test_queries': test_queries,
    'notes': [
        'This is a baseline RAG generation stage without formal evaluation.',
        'Reranker model is English MS MARCO, not Turkish-legal-specific.',
        'Some corpus records may have missing source (NaN) values.',
        'Answer quality may be improved with fine-tuning and stronger models.',
        'Future notebooks: evaluation metrics, answer quality, hallucination behavior.'
    ]
}

metadata_path = f"{RAG_OUTPUT_DIR}/rag_generation_metadata.json"
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Saved metadata: {metadata_path}")
print(f"\nMetadata summary:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# Section 16: Optional - Save Prompt Examples

In [ ]:
# Save prompt examples for inspection
prompt_examples_path = f"{RAG_OUTPUT_DIR}/rag_prompt_examples.txt"

with open(prompt_examples_path, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("RAG Prompt Examples\n")
    f.write("=" * 80 + "\n\n")
    
    for test_idx, query in enumerate(test_queries[:2], 1):
        # Retrieve context
        context = get_final_context(
            query,
            embedding_model,
            faiss_index,
            retrieval_corpus,
            row_mapping,
            reranker,
            top_k=TOP_K_FINAL,
            alpha=ALPHA
        )
        
        # Build prompt
        prompt, sources = build_rag_prompt(query, context)
        
        f.write(f"\n\n{'='*80}\n")
        f.write(f"EXAMPLE {test_idx}\n")
        f.write(f"{'='*80}\n\n")
        f.write(f"Query: {query}\n\n")
        f.write(f"Prompt Sent to Model:\n")
        f.write("-" * 40 + "\n")
        f.write(prompt)
        f.write("\n" + "-" * 40 + "\n")
        f.write(f"\nContext sources used: {sources}\n")

print(f"✓ Saved prompt examples: {prompt_examples_path}")

# Section 17: Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("BASELINE RAG GENERATION - COMPLETE")
print("="*80)

print(f"""
✓ Generated answers for {len(test_queries)} Turkish legal test queries

📁 Saved outputs:
  - {RAG_OUTPUT_DIR}/rag_generated_answers.csv
  - {RAG_OUTPUT_DIR}/rag_generated_answers.jsonl
  - {RAG_OUTPUT_DIR}/rag_generation_metadata.json
  - {RAG_OUTPUT_DIR}/rag_prompt_examples.txt

🔄 Pipeline architecture:
  Query → Dense Embedding + BM25 → Hybrid Fusion → Cross-Encoder Reranking → LLM Generation

⚙️ Configuration used:
  - Embedding model: {EMBEDDING_MODEL_NAME}
  - Reranker model: {RERANKER_MODEL_NAME}
  - Generation model: {GENERATION_MODEL_NAME}
  - Top-K contexts: {TOP_K_FINAL}
  - Max new tokens: {MAX_NEW_TOKENS}
  - Temperature: {TEMPERATURE}

📝 Next steps:

1. **Notebook 07 - Benchmark Evaluation**
   - Compute retrieval metrics (MRR, NDCG, recall@k)
   - Evaluate answer quality (BLEU, ROUGE, semantic similarity)
   - Measure hallucination behavior
   - Analyze citation accuracy

2. **Notebook 08 - Gradio Interface**
   - Create interactive demo for end-to-end RAG system
   - Allow users to input queries and see live results
   - Visualize retrieval pipeline (dense → hybrid → reranked → answer)

3. **Fine-tuning & Optimization**
   - Fine-tune reranker on Turkish legal corpus
   - Fine-tune or prompt-engineer generation model with Turkish legal examples
   - Tune hybrid blend parameter (alpha) per use case
   - Experiment with stronger multilingual generation models

4. **Production Deployment**
   - Build API server (FastAPI)
   - Implement caching for embeddings
   - Add user feedback loop for continuous improvement

⚠️  Important notes:
   - Reranker is English MS MARCO-trained, not Turkish-legal-specific
   - Some corpus records have missing source values (NaN) - may affect citations
   - This is a baseline system - answer quality varies by query
   - Hallucination reduction relies on retrieval quality + grounded prompts
""")

print("="*80)